<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [3]:
import os
import json
import pandas as pd
import pip
import string
import re

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

In [4]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [5]:
# import zipfile
# from multiprocessing import Pool

DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)


We then prepare the entry point for the Spark functionalities that will we use from now on.

In [6]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!rm spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [7]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form.

In [245]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

def extract_data(member):
    rdd = sc.textFile(DATA_DIR + "/" + member + ".csv")

    if member == 'actors':
        rdd = rdd.zipWithIndex().map(lambda r: r[0] + ',' + str(r[1]))
        rdd = rdd.map(lambda r: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', r))
    rdd = rdd.map(lambda r: re.split(r',(?! )', r))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions

    # get column name from csv different from 'id'
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    if member == 'actors':
        column_names[-1] = 'movie_n'
    rdd = (rdd
            .map(lambda r: (r[0], dict(zip(column_names, r[1:]))))
            .filter(lambda r: r[0]!='id'))
    return rdd

for member in members_to_extract:
    letterboxd_RDDs[member] = extract_data(member)


In [246]:
letterboxd_RDDs['themes'] = letterboxd_RDDs['themes'].mapValues(lambda x: {**x, 'theme': x['theme'].strip('"')})

Let's look at the amount of rows for each category.

In [247]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	5798450
Number of rows for crew:	4720183
Number of rows for genres:	1046849
Number of rows for movies:	941597
Number of rows for themes:	125641


A glimpse at the structure of the rows in the RDDs of each category.

In [248]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 ('1000001', {'name': 'Margot Robbie', 'role': 'Barbie', 'movie_n': '1'})
Row for crew:	 ('1000001', {'role': 'Director', 'name': 'Greta Gerwig'})
Row for genres:	 ('1000001', {'genre': 'Comedy'})
Row for movies:	 ('1000001', {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
Row for themes:	 ('1000001', {'theme': 'Humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.

In [249]:
def get_sample(rdd, size):
    return rdd.filter(lambda r: int(r[0],10)<=1000000+size)

for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], 100)

In [250]:
print(letterboxd_RDDs['actors'].count())
print(letterboxd_RDDs['crew'].count())
print(letterboxd_RDDs['genres'].count())
print(letterboxd_RDDs['movies'].count())
print(letterboxd_RDDs['themes'].count())

6130
9115
275
100
704


In [251]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	6130
Number of rows for crew:	9115
Number of rows for genres:	275
Number of rows for movies:	100
Number of rows for themes:	704


For each category the available attributes are the following:

| **Category**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genre                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | theme                               |

For this project we would like to focus on the following features:

| **Category**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genre                               |
| **movies**   | name, date, minute, rating |
| **themes**   | theme                               |


We keep only the 6 most relevant actors in each movie.

In [252]:
n_actors = 6
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], sorted(r[1], key=lambda x: x["movie_n"])[:n_actors])))

In [253]:
def filter_dict_fields(d, final_fields):
    return {key: d[key] for key in final_fields if key in d}

def remove_dict_field(d, field):
    del d[field]
    return d

def rename_key_in_dict(d, old_key, new_key):
    d[new_key] = d.pop(old_key)
    return d

def replace_dict_values(d, keys_to_replace, f):
    return {k: (f(v) if k in keys_to_replace else v) for k, v in d.items()}

In [254]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], {key: [d[key] for d in r[1]] for key in r[1][0]}))
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'actors'))))

We filter only the directors from the crew dataset, and we ony keep their name.

In [255]:
letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                           .filter(lambda r: r[1]['role']=='Director')
                           .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                           .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'director'))))

For each movie we only store the attributes listed in the table above.

In [256]:
letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name', 'date', 'minute', 'rating']))))

In [257]:
for member in members_to_extract:
    print(letterboxd_RDDs[member].first())
    print(letterboxd_RDDs[member].count())

('1000004', {'actors': ['Edward Norton', 'Brad Pitt', 'Helena Bonham Carter', 'Meat Loaf', 'Jared Leto', 'Zach Grenier']})
100
('1000001', {'director': 'Greta Gerwig'})
109
('1000001', {'genre': 'Comedy'})
275
('1000001', {'name': 'Barbie', 'date': '2023', 'minute': '114', 'rating': '3.86'})
100
('1000001', {'theme': 'Humanity and the world around us'})
704


In [258]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.join(letterboxd_RDDs[member]).mapValues(lambda x: {**x[0], **x[1]})

In [259]:
movies_RDD.first()

('1000064',
 {'actors': ['Amy Adams',
   'Jeremy Renner',
   'Forest Whitaker',
   'Michael Stuhlbarg',
   'Tzi Ma',
   "Mark O'Brien"],
  'director': 'Denis Villeneuve',
  'genre': 'Science Fiction',
  'name': 'Arrival',
  'date': '2016',
  'minute': '116',
  'rating': '4.12',
  'theme': 'Monsters, aliens, sci-fi and the apocalypse'})

# Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating each movie data dictionary into a vector in $\mathbb{R}^{17}$. In fact we will work in a space where each feature is a dimension, and we consider each of the 10 actors starring in the movie as a distinct attribute.



In [260]:
def list_to_dict(l, field_base_name):
    return {f"{field_base_name}{i + 1}": value for i, value in enumerate(l)}

movies_RDD = (movies_RDD.map(lambda r: (r[0], {**r[1],**list_to_dict(r[1]['actors'], 'actor')}))
                                        .map(lambda r: (r[0], remove_dict_field(r[1], 'actors'))))

In [261]:
letterboxd_RDDs['actors'].first()

('1000004',
 {'actors': ['Edward Norton',
   'Brad Pitt',
   'Helena Bonham Carter',
   'Meat Loaf',
   'Jared Leto',
   'Zach Grenier'],
  'actor1': 'actors'})

Below we list the data types of the various attributes.

| **Attributes**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actors**    | string         |
| **director**     | string                        |
| **genre**   | string                             |
| **theme**   | string                             |
| **name**   | string |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into strings to be able to work in an Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


## String preprocessing

We bring all the strings in data dictionary to lower case, eccept for the names of actors and director. In addition to names always being capitalized, we will not consider them from a semantic point of view, so their uniformation in preparation for the next steps would be useless.

In [ ]:
categories_lower = ['genre', 'name', 'theme']
    movies_lower_RDD = movies_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_lower, lambda x: x.lower())))

To distill the semantics of the movie's theme we apply the following NLP processing steps:
* remove stop words
* replace the words with their lemmatized version

In [ ]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

In [159]:
from functools import reduce

combine_functions = lambda *funcs: lambda x: reduce(lambda v, f: f(v), funcs, x) #v accumulator value
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

nlp_processing = combine_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)

categories_nlp = ['theme']
movies_RDD = movies_lower_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_nlp, nlp_processing)))

In [ ]:
movies_RDD.take(5)

### Embedding strings using word2vec

We apply word2vec embeddings to attributes:
* name
* genre
* theme

In [218]:
embedding_func = lambda x: nlp(x).vector

categories_word2vec = ['name', 'genre', 'theme']
movie_cat_embeddings_word2vec_RDD = movies_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_word2vec, embedding_func)))

In [ ]:
movie_cat_embeddings_word2vec_RDD.take(1)

### Hashing strings

We hash the strings of the features:
* actors
* director
person name hash
(no bias towards similar names)

In [228]:
import hashlib
hash_func = lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']
movie_cat_embeddings_RDD = movie_cat_embeddings_word2vec_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_hash, hash_func)))

## Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

In [229]:
movie_cat_embeddings_RDD.first()

('1000064',
 {'director': 29477009849425759340612412694934903989,
  'genre': array([-0.27108002, -3.68004   ,  1.7981    , -2.67355   ,  0.20334502,
         -0.6242425 ,  6.10215   ,  4.98125   , -0.547055  , -2.27563   ,
          3.6404    ,  4.2419996 , -3.8442998 , -0.69453   ,  2.3896499 ,
          2.97094   ,  4.98995   ,  3.83935   , -3.0113351 ,  1.8192    ,
          1.77776   ,  0.322445  , -5.1607504 ,  1.9984    , -1.078565  ,
         -3.42015   ,  0.42837998, -2.41395   , -0.952133  ,  4.7316    ,
         -1.015375  ,  2.1853251 ,  0.577665  , -1.16634   , -5.4253    ,
         -0.2707    ,  3.39715   , -2.4335501 , -1.51835   ,  0.17844999,
         -0.35981998, -0.98864996, -0.14400005,  1.90015   ,  1.0259199 ,
          1.0407801 , -1.2413181 , -2.54155   ,  1.83139   , -1.03794   ,
         -3.1072502 ,  3.4145498 , -0.42069   , -0.13498999,  0.20299995,
         -0.19891   , -3.60495   ,  2.0658002 ,  2.86704   ,  1.3234501 ,
         -0.09850001, -0.58669996, -2

In [ ]:
movie_vectors_RDD = movie_cat_embeddings_RDD.map(lambda r: (r[0], list(r[1].values())))

In [ ]:
movie_vectors_RDD.first()

## LSH hash functions

We build the locality sensitive family $\mathbf{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathbf{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathbf{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathbf{F}$ as hash functions.

Since we will compute the hash functions of each of the elements in the dataset, given each of the hash functions in $\mathbf{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathbf{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.